# Selecting answers from Scorio Lite with `scorio.aggregate`

This notebook chooses one answer from each 80-attempt candidate pool. The input arrays have
shape `M x N`: questions by candidates. Answer labels come from `extracted_answer`,
verifier scores come from CompassVerifier-3B or the reference-free verifier, and confidence
comes from completion log probabilities.

|  |  |
| --- | --- |
| module | [`scorio/aggregate`](https://github.com/mohsenhariri/scorio/tree/main/scorio/aggregate) |
| method reference | [`scorio/aggregate/README.md`](https://github.com/mohsenhariri/scorio/blob/main/scorio/aggregate/README.md) |
| dataset | [Scorio Lite](https://huggingface.co/datasets/harimo/scorio-lite) |
| install | `pip install scorio` |

In [1]:
import numpy as np
import pandas as pd
from datasets import load_dataset
from IPython.display import display

from scorio import agg, eval

repo_name = "harimo/scorio-lite"
model_name = "gpt-oss-20b_medium"
task = "aime_2026"

rows = (load_dataset(repo_name, "meta-math", split=task)
        .flatten()
        .filter(lambda row: row["model_key"] == model_name)
        .select_columns([
            "data_id", "seed", "extracted_answer", "evalscope_is_correct",
            "cv3b_abc_A", "llmv_problem_understanding_expected",
            "llmv_reasoning_validity_expected", "llmv_conclusion_support_expected",
            "tokens.completion_avg_logprob",
        ])
        .to_pandas()
        .sort_values(["data_id", "seed"]))

M, N = 30, 80

answers = rows.extracted_answer.to_numpy().reshape(M, N).astype(object)
answers[answers == "NotFound"] = None

# CompassVerifier's normalized probability for label A ("correct")
cv3b = rows.cv3b_abc_A.to_numpy().reshape(M, N)

# The reference-free verifier's mean 1-20 score, converted to its documented [0, 1] reward
llmv_expected = (
    rows.llmv_problem_understanding_expected
    + rows.llmv_reasoning_validity_expected
    + rows.llmv_conclusion_support_expected
).to_numpy().reshape(M, N) / 3
llmv = (llmv_expected - 1) / 19

confidence = rows["tokens.completion_avg_logprob"].to_numpy().reshape(M, N)

# A selected label is correct when EvalScope marked that label correct for the question.
accepted = [
    set(group.loc[group.evalscope_is_correct.astype(bool), "extracted_answer"])
    for _, group in rows.groupby("data_id", sort=True)
]

print(answers.shape, cv3b.shape, llmv.shape, confidence.shape)

Filter:   0%|          | 0/9600 [00:00<?, ? examples/s]

(30, 80) (30, 80) (30, 80) (30, 80)


## One question, eight candidates

For question 24, the first sample is wrong and the next seven are correct. The verifier
assigns almost all of its A/B/C probability to the correct label.

In [2]:
question_id = 24
pool = answers[question_id, :8]
scores = cv3b[question_id, :8]

print("candidates    ", list(pool))
print("P(correct)    ", scores.round(3))
print()
print("first sample  ", pool[0])
print("majority_vote ", agg.majority_vote(pool))
print("best_of_n     ", agg.best_of_n(pool, scores))
print("accepted      ", accepted[question_id])

candidates     ['950', '850', '850', '850', '850', '850', '850', '850']
P(correct)     [0. 1. 1. 1. 1. 1. 1. 1.]

first sample   950
majority_vote  850
best_of_n      850
accepted       {'850'}


## All 30 questions

Selection methods return answers, not accuracy. The helper below checks the selected label
against EvalScope's grading for the same question and then uses `scorio.eval` for the
point estimate and standard error.

In [3]:
def accuracy(selected):
    hit = np.array([[int(answer in accepted[i])]
                    for i, answer in enumerate(selected)])
    mu, sigma = eval.avg(hit)
    return {"accuracy": round(float(mu), 3), "sigma": round(float(sigma), 3)}


n = 8
A, V, L, C = answers[:, :n], cv3b[:, :n], llmv[:, :n], confidence[:, :n]

results = {
    "first sample": accuracy(A[:, 0]),
    "majority_vote": accuracy(agg.majority_vote(A)),
    "weighted_majority_vote (CV3B)": accuracy(agg.weighted_majority_vote(A, V)),
    "best_of_n (CV3B)": accuracy(agg.best_of_n(A, V)),
    "best_of_n (reference-free verifier)": accuracy(agg.best_of_n(A, L)),
    "filtered_vote (logprob)": accuracy(agg.filtered_vote(A, C, keep=0.5, weighted=False)),
    "rank_weighted_vote (logprob)": accuracy(agg.rank_weighted_vote(A, C)),
}

display(pd.DataFrame(results).T.sort_values("accuracy", ascending=False))

,accuracy,sigma
majority_vote,0.900,0.129
weighted_majority_vote (CV3B),0.900,0.129
best_of_n (CV3B),0.900,0.129
rank_weighted_vote (logprob),0.900,0.129
best_of_n (reference-free verifier),0.900,0.129
first sample,0.867,0.129
filtered_vote (logprob),0.867,0.129


Mean log probability is negative, so it should not be passed directly to a rule that
sums scores as vote weights. `filtered_vote(..., weighted=False)` and
`rank_weighted_vote` use only its ordering.

## Increasing the sample budget

In [4]:
budgets = [1, 2, 4, 8, 16, 32, 80]

sweep = pd.DataFrame({
    n: {
        "mean sample": round(float(np.mean([
            [int(a in accepted[i]) for a in answers[i, :n]]
            for i in range(M)
        ])), 3),
        "majority_vote": accuracy(agg.majority_vote(answers[:, :n]))["accuracy"],
        "weighted (CV3B)": accuracy(
            agg.weighted_majority_vote(answers[:, :n], cv3b[:, :n])
        )["accuracy"],
        "best_of_n (CV3B)": accuracy(
            agg.best_of_n(answers[:, :n], cv3b[:, :n])
        )["accuracy"],
    }
    for n in budgets
}).T
sweep.index.name = "samples"

display(sweep)

,mean sample,majority_vote,weighted (CV3B),best_of_n (CV3B)
samples,,,,
1,0.867,0.867,0.867,0.867
2,0.800,0.867,0.900,0.900
4,0.792,0.900,0.900,0.900
8,0.800,0.900,0.900,0.900
16,0.787,0.900,0.933,0.933
32,0.791,0.900,0.967,0.967
80,0.777,0.933,0.967,0.967


## Stopping early

These rules inspect the answer stream and stop when later samples are unlikely to change
the vote. Their sample use and final accuracy should both be reported.

In [5]:
def run_until_stop(should_stop):
    used, selected = [], []
    for i in range(M):
        stop = N
        for t in range(2, N + 1):
            if should_stop(answers[i, :t]):
                stop = t
                break
        used.append(stop)
        selected.append(agg.majority_vote(answers[i, :stop]))
    return round(float(np.mean(used)), 1), accuracy(selected)["accuracy"]


print("all 80 samples       ",
      (80.0, accuracy(agg.majority_vote(answers))["accuracy"]))
print("adaptive_consistency ",
      run_until_stop(lambda seen: agg.adaptive_consistency_stop(seen, threshold=0.95)))
print("esc (window of 3)    ",
      run_until_stop(lambda seen: len(seen) >= 3 and agg.esc_stop(seen[-3:])))

all 80 samples        (80.0, 0.933)
adaptive_consistency  (11.4, 0.933)
esc (window of 3)     (10.8, 0.933)


## Confidence from token log probabilities

The meta config stores aggregate log-probability values. Per-model configs also store the
token lists, so Scorio can recompute the same signals.

In [6]:
full_pool = (load_dataset(repo_name, f"{model_name}-math", split=task)
             .filter(lambda row: row["data_id"] == question_id)
             .sort("seed")
             .select(range(3)))

for attempt in full_pool:
    logprobs = attempt["tokens"]["completion_logprob_list"]
    stored = attempt["tokens"]
    print(
        f"seed {attempt['seed']}  "
        f"mean_logprob {agg.mean_logprob(logprobs):.6f} "
        f"(stored {stored['completion_avg_logprob']:.6f})  "
        f"perplexity {agg.perplexity(logprobs):.4f} "
        f"(stored {stored['completion_ppl']:.4f})"
    )

Filter:   0%|          | 0/2400 [00:00<?, ? examples/s]

seed 0  mean_logprob -0.689930 (stored -0.689930)  perplexity 1.9936 (stored 1.9936)
seed 1  mean_logprob -0.813660 (stored -0.813660)  perplexity 2.2561 (stored 2.2561)
seed 2  mean_logprob -0.633974 (stored -0.633974)  perplexity 1.8851 (stored 1.8851)


Scorio Lite stores the realized token's log probability and rank, but not top-20
candidate distributions. Signals such as token entropy, self-certainty, and log-probability
margin require `harimo/scorio-math` or `harimo/scorio-gpqa`.

The [aggregation reference](https://github.com/mohsenhariri/scorio/blob/main/scorio/aggregate/README.md)
maps published methods to their confidence signal and selection rule.